<a href="https://colab.research.google.com/github/looooore/Pytorch_Colab/blob/main/CNN.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader
from torchvision import datasets, transforms
from torchvision.utils import make_grid

import numpy as np
import pandas as pd
from sklearn.metrics import confusion_matrix
import matplotlib.pyplot as plt
%matplotlib inline

In [2]:
# Convert MNIST Image Files into a Tensor of 4-Dimensions (# of Images, Height, Color, Channel)
transform = transforms.ToTensor()

In [3]:
# Train Data
train_data = datasets.MNIST(root='/cnn_data', train=True, download=True, transform=transform)

100%|██████████| 9.91M/9.91M [00:00<00:00, 17.4MB/s]
100%|██████████| 28.9k/28.9k [00:00<00:00, 487kB/s]
100%|██████████| 1.65M/1.65M [00:00<00:00, 3.84MB/s]
100%|██████████| 4.54k/4.54k [00:00<00:00, 7.27MB/s]


In [4]:
# Test Data
test_data = datasets.MNIST(root='/cnn_data', train=False, download=True, transform=transform)

In [5]:
test_data

Dataset MNIST
    Number of datapoints: 10000
    Root location: /cnn_data
    Split: Test
    StandardTransform
Transform: ToTensor()

In [6]:
train_data

Dataset MNIST
    Number of datapoints: 60000
    Root location: /cnn_data
    Split: Train
    StandardTransform
Transform: ToTensor()

In [7]:
# Create a small batch size for images... let's say 10
train_loader = DataLoader(train_data, batch_size=10, shuffle=True)
test_loader = DataLoader(test_data, batch_size=10, shuffle=False)

In [8]:
# Define Our CNN Model
# Describe convolutional layer and what it's doing (2 convolucional layers)
# This is just an exemple i the next video we'll build out the actual model
conv1 = nn.Conv2d(1, 6, 3, 1)
conv2 = nn.Conv2d(6, 16, 3, 1)

In [9]:
conv1

Conv2d(1, 6, kernel_size=(3, 3), stride=(1, 1))

In [10]:
# Grab 1 MNIST record/image
for i, (X_Train, y_train) in enumerate(train_data):
  break

In [11]:
X_Train.shape # 1 img 28x28

torch.Size([1, 28, 28])

In [12]:
x = X_Train.view(1,1,28,28)

In [13]:
# Perform our first convolutional
x = F.relu(conv1(x)) # Rectified Linear Unit for our actvation funcation

In [14]:
# 1 single image, 6 is the filters we asked for, 26x26
x.shape # Matriz de 1 por 6 por 26 por 26

torch.Size([1, 6, 26, 26])

In [16]:
# pass thru the pooling layer
x = F.max_pool2d(x, 2,2) # kernel of 2 and stride of 2

In [17]:
x.shape # 26 / 2 = 13

torch.Size([1, 6, 13, 13])

In [18]:
# Do our second convolucional layer
x = F.relu(conv2(x))

In [19]:
x.shape # We didn't set padding so we lose 2 pixels around the outside of the image

torch.Size([1, 16, 11, 11])

In [20]:
# Pooling layer
x = F.max_pool2d(x, 2,2)

In [21]:
x.shape # 11 / 2 = 5.5 but we have to round down, because you can't invent data to round up

torch.Size([1, 16, 5, 5])

In [26]:
pooling_process = ((28 - 2) / 2 - 2) / 2 # = 5.5
int(pooling_process)

5

In [28]:
# Model Class
class ConvolutionalNetwork(nn.Module):
  def __init__(self):
    super().__init__()
    self.conv1 = nn.Conv2d(1,6,3,1)
    self.conv2 = nn.Conv2d(6,16,3,1)
    # Fully Connect Layer
    self.fc1 = nn.Linear(5*5*16, 120) # 120 neurônios
    self.fc2 = nn.Linear(120, 84)
    self.fc3 = nn.Linear(84, 10)

  def forward(self, x):
    X = F.relu(self.conv1(X))
    X = F.max_pool2d(X, 2,2) # 2x2 Kernal and Stride 2
    # Scond Pass (2 pooling layers)
    X = F.relu(self.conv2(X))
    X = F.max_pool2d(X, 2,2)

    # Re-View to flatten it ou
    X = X.view(-1, 16*5*5) # Negative one so that we can vary the batch size

    # Fully Connect Layers
    X = F.relu(self.fc1(X))
    X = F.relu(self.fc2(X))
    X = self.fc3(X)
    return F.log_softmax(X, dim=1)

In [30]:
# Create an Instance of our Model
torch.manual_seed(41)
model = ConvolutionalNetwork()
model

ConvolutionalNetwork(
  (conv1): Conv2d(1, 6, kernel_size=(3, 3), stride=(1, 1))
  (conv2): Conv2d(6, 16, kernel_size=(3, 3), stride=(1, 1))
  (fc1): Linear(in_features=400, out_features=120, bias=True)
  (fc2): Linear(in_features=120, out_features=84, bias=True)
  (fc3): Linear(in_features=84, out_features=10, bias=True)
)

In [31]:
# Loss Function Optimizer
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001) # Smaller the Learning Rate, longer its gonna take to train.